# Brief recap of the data, goals, and tasks

This project uses NFL play by play data from the 2019 season. The raw dataset is play level and includes many play types and fields, so I filtered it to focus on offensive decision making. The cleaned dataset includes only run and pass plays on first, second, and third down, along with key context variables like yards to go and field position. To make the data easier to interpret, I created a distance bucket feature (Short, Medium, Long) and a red zone indicator (within 20 yards of the end zone).

The goal of the project is to understand how teams make play calling decisions and whether those choices align with efficiency. I focused the narrative on the Baltimore Ravens because team level comparisons can get cluttered when the viewer is asked to track 32 teams across multiple situations. The questions I designed around were: how the Ravens shift run versus pass based on down and distance, how the Ravens compare to the rest of the league on passing tendency and average yards gained, and how Ravens play calling changes in the red zone compared to the rest of the field.

These goals map to three concrete tasks that guided the visual design. Task 1 is to compare run versus pass share across downs and distance buckets for the Ravens. Task 2 is to place the Ravens in league context by comparing all teams on pass rate and average yards per play. Task 3 is to measure how play calling shifts between red zone and non red zone situations for the Ravens.

# Data Preparation
## Load and Clean the 2019 NFL Play by Play Data

This notebook prepares a clean, easy to understand version of the NFL play by play dataset for the 2019 season. The raw data contains many columns and many play types, so the first step is to filter it down to the specific play types we care about (runs and passes) and remove rows that do not have the core values needed for analysis. This will make the dataset easier to explain and will also reduce noise before we aggregate it for visualization.

In [1]:
import altair as alt
import pandas as pd

url = "https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_2019.parquet"

pbp_2019 = pd.read_parquet(url)

pbp_clean = pbp_2019[
    (pbp_2019["play_type"].isin(["run", "pass"])) &
    (pbp_2019["down"].isin([1, 2, 3])) &
    (pbp_2019["posteam"].notna()) &
    (pbp_2019["yards_gained"].notna())
].copy()

pbp_clean = pbp_clean.rename(columns={
    "posteam": "team",
    "ydstogo": "yards_to_go",
    "yardline_100": "yards_to_endzone"
})

print(pbp_clean.shape)
pbp_clean.head()

(32776, 372)


,play_id,game_id,old_game_id,home_team,away_team,season_type,week,team,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
2,51.0,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0.0,0.0,-1.658763,NaN,NaN,NaN,NaN,NaN,0.486799,51.320082
3,79.0,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0.0,0.0,-0.538914,NaN,NaN,NaN,NaN,NaN,0.639994,-63.999379
4,100.0,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,ATL,away,MIN,...,0.0,0.0,0.142138,NaN,NaN,NaN,NaN,NaN,0.933516,6.648362
7,185.0,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,MIN,home,ATL,...,1.0,0.0,0.258130,0.747668,6.732305,6.0,0.505931,0.053389,0.705088,29.491204
8,214.0,2019_01_ATL_MIN,2019090804,MIN,ATL,REG,1,MIN,home,ATL,...,0.0,0.0,2.785377,0.304460,2.208216,0.0,0.999368,0.484263,0.803992,19.600838


## Select Relevant Columns

The original dataset contains many variables that are not needed for this analysis. To improve clarity and reduce unnecessary complexity, I limit the dataset to only the fields required for the visualizations.

These include:
- team
- down
- yards to go
- play type
- yards gained
- field position (distance to end zone)

In [2]:
pbp_clean = pbp_clean[[
    "team",
    "down",
    "yards_to_go",
    "yards_to_endzone",
    "play_type",
    "yards_gained"
]]

print(pbp_clean.shape)
pbp_clean.head()

(32776, 6)


,team,down,yards_to_go,yards_to_endzone,play_type,yards_gained
2,ATL,1.0,10.0,75.0,pass,-8.0
3,ATL,2.0,18.0,83.0,run,4.0
4,ATL,3.0,14.0,79.0,run,12.0
7,MIN,1.0,20.0,31.0,pass,8.0
8,MIN,2.0,12.0,23.0,pass,23.0


## Feature Engineering

To make the dataset easier to interpret, I create a few derived variables that better reflect how football decisions are typically analyzed.

First, I convert yards to go into distance categories (short, medium, long), which are easier to interpret than raw numeric values. Second, I create a red zone indicator to capture plays occurring within 20 yards of the end zone. Finally, I standardize play type labels to improve readability in visualizations.

In [3]:
def distance_bucket(x):
    if x <= 3:
        return "Short"
    elif x <= 7:
        return "Medium"
    else:
        return "Long"

pbp_clean["distance"] = pbp_clean["yards_to_go"].apply(distance_bucket)

pbp_clean["red_zone"] = pbp_clean["yards_to_endzone"] <= 20

pbp_clean["play_type"] = pbp_clean["play_type"].replace({
    "run": "Run",
    "pass": "Pass"
})

pbp_clean.head()

,team,down,yards_to_go,yards_to_endzone,play_type,yards_gained,distance,red_zone
2,ATL,1.0,10.0,75.0,Pass,-8.0,Long,False
3,ATL,2.0,18.0,83.0,Run,4.0,Long,False
4,ATL,3.0,14.0,79.0,Run,12.0,Long,False
7,MIN,1.0,20.0,31.0,Pass,8.0,Long,False
8,MIN,2.0,12.0,23.0,Pass,23.0,Long,False


## Aggregate Data for Visualization

Because the dataset is at the play level, it contains more detail than is needed for the visualizations. To make the data more interpretable and suitable for Altair, I aggregate it into summary tables aligned with the key analytical tasks.

These aggregations support three main visualizations:
1. Play calling tendencies by situation (down and distance)
2. Team level efficiency (pass rate vs average yards gained)
3. Red zone play calling behavior

This step reduces the dataset size while preserving the patterns needed for analysis.

### 1. Play calling tendencies

In [4]:
play_call = pbp_clean.groupby(
    ["team", "down", "distance", "play_type"]
).size().reset_index(name="plays")

print(play_call.shape)
play_call.head()

(574, 5)


,team,down,distance,play_type,plays
0,ARI,1.0,Long,Pass,218
1,ARI,1.0,Long,Run,202
2,ARI,1.0,Medium,Pass,7
3,ARI,1.0,Medium,Run,1
4,ARI,1.0,Short,Pass,3


### 2. Team Efficiency Summary

To evaluate whether certain play calling strategies are more effective, I compute two metrics at the team level:
- Pass rate (percentage of plays that are passes)
- Average yards gained per play

These metrics will be used to compare tendencies and outcomes across teams.

In [5]:
team_summary = pbp_clean.groupby("team").agg(
    pass_rate=("play_type", lambda x: (x == "Pass").mean()),
    avg_yards=("yards_gained", "mean")
).reset_index()

print(team_summary.shape)
team_summary.head()

(32, 3)


,team,pass_rate,avg_yards
0,ARI,0.607254,5.583420
1,ATL,0.674528,5.613208
2,BAL,0.464545,6.241818
3,BUF,0.555137,5.388313
4,CAR,0.643403,5.184512


### 3. Red Zone Summary

Finally, I isolate red zone plays to examine how teams adjust their play calling in high leverage situations. This aggregation compares run versus pass frequency within the red zone for each team.

In [6]:
red_zone = pbp_clean[pbp_clean["red_zone"] == True].copy()

red_zone_summary = red_zone.groupby(
    ["team", "play_type"]
).size().reset_index(name="plays")

print(red_zone_summary.shape)
red_zone_summary.head()

(64, 3)


,team,play_type,plays
0,ARI,Pass,79
1,ARI,Run,62
2,ATL,Pass,97
3,ATL,Run,60
4,BAL,Pass,76


# Visualizations

## Design for Task 1:

**Chart 1: Ravens run versus pass by distance and down.** I used normalized stacked bars with distance on the x axis and play type as the stacked segments. This choice makes the run-pass split readable at a glance, and because each bar sums to 100 percent it supports comparison across distance buckets without the viewer needing to account for different play counts. I faceted by down using side by side panels so the viewer can compare how the same distance bucket behaves on different downs. I also used direct down labels (1st Down, 2nd Down, 3rd Down) and removed redundant axis titles to reduce clutter. This chart is designed to answer a simple question quickly: in a given situation, do the Ravens lean run or pass.


<img src="Chart 1.jpeg" width="700">

In [7]:
import warnings
warnings.filterwarnings("ignore")

team_name = "BAL"

pbp_team = pbp_clean[pbp_clean["team"] == team_name].copy()

distance_order = ["Short", "Medium", "Long"]
pbp_team["distance"] = pd.Categorical(pbp_team["distance"], categories=distance_order, ordered=True)
pbp_clean["distance"] = pd.Categorical(pbp_clean["distance"], categories=distance_order, ordered=True)

down_labels = {1: "1st Down", 2: "2nd Down", 3: "3rd Down"}
pbp_team["down_label"] = pbp_team["down"].map(down_labels)

play_call_team = pbp_team.groupby(
    ["team", "down", "down_label", "distance", "play_type"]
).size().reset_index(name="plays")

play_call_team["pct"] = play_call_team["plays"] / play_call_team.groupby(
    ["team", "down", "down_label", "distance"]
)["plays"].transform("sum")

x_label = (
    alt.Chart(pd.DataFrame({"label": ["Distance to Go"]}))
    .mark_text(fontSize=18, fontWeight="bold")
    .encode(text="label:N")
    .properties(height=30, width=220 * 3 + 40)
)

down_labels = {1: "1st Down", 2: "2nd Down", 3: "3rd Down"}
play_call_team["down_label"] = play_call_team["down"].map(down_labels)

base = (
    alt.Chart(play_call_team)
    .mark_bar()
    .encode(
        x=alt.X("distance:N", sort=distance_order, title=None),
        y=alt.Y("pct:Q", stack="normalize", title="Share of Plays", axis=alt.Axis(format="%")),
        color=alt.Color(
            "play_type:N",
            sort=["Run", "Pass"],
            title="Play Type",
            legend=alt.Legend(titleFontSize=14, labelFontSize=12, symbolSize=150)
        ),
        column=alt.Column(
            "down_label:N",
            title=None,
            sort=["1st Down", "2nd Down", "3rd Down"],
            header=alt.Header(labelFontSize=14)
        ),
        tooltip=[
            alt.Tooltip("team:N", title="Team"),
            alt.Tooltip("down_label:N", title="Down"),
            alt.Tooltip("distance:N", title="Distance"),
            alt.Tooltip("play_type:N", title="Play Type"),
            alt.Tooltip("plays:Q", title="Plays"),
            alt.Tooltip("pct:Q", title="Share", format=".1%")
        ]
    )
    .properties(width=220, height=300)
)

chart1 = (
    alt.vconcat(base, x_label, spacing=5)
    .properties(
        title=alt.TitleParams(
            text="Baltimore Ravens Run vs Pass by Distance and Down",
            anchor="middle",
            fontSize=18
        )
    )
    .configure_axis(labelFontSize=11, titleFontSize=14)
)

chart1

alt.VConcatChart(...)

## Design for Task 2:

**Chart 2: League comparison of pass rate versus efficiency.** A scatter plot is the most direct way to compare two quantitative variables because position is a strong visual channel. Each point represents a team, with the x axis as pass rate and the y axis as average yards gained per play. I highlighted the Ravens using a purple mark and a direct label, while keeping the rest of the league in a neutral color. This removes the need for a legend and makes the focal team obvious. I also constrained the axes to the range where teams actually sit so the pattern is readable instead of compressed into a corner. This chart supports task 2 by showing whether the Ravens are an outlier in strategy, efficiency, or both.

<img src="Chart 2.jpeg" width="700">

In [8]:
team_summary_all = pbp_clean.groupby("team").agg(
    pass_rate=("play_type", lambda x: (x == "Pass").mean()),
    avg_yards=("yards_gained", "mean")
).reset_index()

team_summary_all["is_ravens"] = team_summary_all["team"].eq(team_name)

x_min = max(0, team_summary_all["pass_rate"].min() - 0.03)
x_max = min(1, team_summary_all["pass_rate"].max() + 0.03)

y_min = team_summary_all["avg_yards"].min() - 0.3
y_max = team_summary_all["avg_yards"].max() + 0.3

base_points = alt.Chart(team_summary_all).mark_circle(size=100, opacity=0.7).encode(
    x=alt.X(
        "pass_rate:Q",
        title="Pass Rate",
        axis=alt.Axis(format="%"),
        scale=alt.Scale(domain=[x_min, x_max])
    ),
    y=alt.Y(
        "avg_yards:Q",
        title="Average Yards per Play",
        scale=alt.Scale(domain=[y_min, y_max])
    ),
    color=alt.condition(
        alt.datum.is_ravens,
        alt.value("#4B0082"),   
        alt.value("#9e9e9e")    
    ),
    tooltip=[
        alt.Tooltip("team:N", title="Team"),
        alt.Tooltip("pass_rate:Q", title="Pass Rate", format=".1%"),
        alt.Tooltip("avg_yards:Q", title="Avg Yards", format=".2f")
    ]
)

ravens_label = (
    alt.Chart(team_summary_all[team_summary_all["is_ravens"] == True])
    .mark_text(
        align="left",
        dx=10,
        dy=-10,
        fontSize=13,
        fontWeight="bold",
        color="purple"
    )
    .encode(
        x="pass_rate:Q",
        y="avg_yards:Q",
        text=alt.value("Ravens")
    )
)

chart2 = (
    (base_points + ravens_label)
    .properties(
        title=alt.TitleParams(
            text="NFL Team Passing Tendencies vs Efficiency (2019 Season)",
            anchor="middle",
            fontSize=18
        ),
        width=600,
        height=420
    )
    .configure_axis(labelFontSize=11, titleFontSize=14)
)

chart2

alt.LayerChart(...)

## Design for Task 3:

**Chart 3: Ravens red zone versus non-red zone play calling shift.** I used a dumbbell chart to emphasize change between two conditions. Each play type is a row, with a point for the non red zone share and a point for the red zone share, connected by a line. The distance between the points is the story, it shows how much play calling shifts when the context changes. I colored the red zone point in red and used a contrasting color for non red zone to make the meaning intuitive without reading a caption. I also removed the y axis labels and placed the Run and Pass labels directly above their dumbbells so the viewer can stay focused on the comparison. Finally, I zoomed the x axis to the relevant range so small but meaningful differences are visible.

Across all charts, the main design goal was to keep the visuals interpretable without requiring football expertise beyond basic concepts like downs, distance to go, and the red zone. The charts use aggregation so the viewer sees patterns rather than noise, and the styling choices were made to reduce clutter and support quick comparison.

<img src="Chart 3.jpeg" width="700">

In [9]:
pbp_team["zone"] = pbp_team["red_zone"].map({True: "Red Zone", False: "Non-Red Zone"})

ravens_zone_summary = pbp_team.groupby(
    ["zone", "play_type"]
).size().reset_index(name="plays")

ravens_zone_summary["pct"] = ravens_zone_summary["plays"] / ravens_zone_summary.groupby(
    ["zone"]
)["plays"].transform("sum")

x_enc = alt.X(
    "pct:Q",
    title="Share of Plays",
    axis=alt.Axis(format="%", tickCount=5, labelFontSize=11, titleFontSize=14),
    scale=alt.Scale(domain=[0.40, 0.60])
)

y_enc = alt.Y(
    "play_type:N",
    title=None,
    axis=None,
    sort=["Run", "Pass"]
)

lines = alt.Chart(ravens_zone_summary).mark_line(
    strokeWidth=3,
    opacity=0.5
).encode(
    y=y_enc,
    x=x_enc,
    detail="play_type:N",
    color=alt.value("#888888")
)

points = alt.Chart(ravens_zone_summary).mark_circle(
    size=220,
    stroke="white",
    strokeWidth=2
).encode(
    y=y_enc,
    x=x_enc,
    color=alt.Color(
        "zone:N",
        sort=["Non-Red Zone", "Red Zone"],
        scale=alt.Scale(
            domain=["Non-Red Zone", "Red Zone"],
            range=["#4C78A8", "#E45756"]  
        ),
        title=None,
        legend=alt.Legend(
    orient="top-left",     
    direction="vertical",
    offset=20,             
    labelFontSize=12,
    symbolSize=140
)
    ),
    tooltip=[
        alt.Tooltip("zone:N", title="Context"),
        alt.Tooltip("play_type:N", title="Play Type"),
        alt.Tooltip("plays:Q", title="Plays"),
        alt.Tooltip("pct:Q", title="Share", format=".1%")
    ]
)

label_data = ravens_zone_summary.groupby("play_type", as_index=False).agg(
    mid_pct=("pct", "mean")
)

labels = alt.Chart(label_data).mark_text(
    align="center",
    dy=-15,
    fontSize=13,
    fontWeight="bold"
).encode(
    y=alt.Y("play_type:N", sort=["Run", "Pass"]),
    x="mid_pct:Q",
    text="play_type:N"
)

chart3 = (
    (lines + points + labels)
    .properties(
        title=alt.TitleParams(
            text="Ravens Red Zone vs Non-Red Zone Play Calling Shift",
            anchor="middle",
            fontSize=18
        ),
        width=650,
        height=240
    )
    .configure_view(stroke=None)
    .configure_axis(grid=True, gridOpacity=0.15)
)

chart3

alt.LayerChart(...)

## Evaluation approach and results

I conducted a small formative evaluation with three participants to test whether the visualizations were understandable and whether the intended insights were easy to recover. The participants were: a coworker who follows football casually, a friend who watches football regularly, and a family member with limited football knowledge. Each participant reviewed the three charts in order and completed the same short set of tasks.

The tasks were: for Chart 1, describe how play calling changes as distance increases on each down and identify one situation where the Ravens are clearly run heavy or pass heavy. For Chart 2, explain whether the Ravens look above or below average in efficiency and whether their pass rate seems typical or unusual. For Chart 3, state whether the Ravens become more run oriented or pass oriented in the red zone and estimate the direction of the change for both play types.

Overall, the participants correctly answered the high level questions with minimal prompting. Chart 1 was rated as the most intuitive because the stacked bars made the run pass split obvious and the side by side down panels encouraged comparison. Chart 2 was understood quickly, but one participant initially assumed the axes were total passing yards and total yards. After that feedback, I made sure the axis titles explicitly say pass rate and average yards per play. Chart 3 was generally liked, and the red color for red zone made the encoding immediately clear. One participant asked what the connecting line meant, and after a short explanation they said it helped them focus on the size of the shift. To address that, I positioned the Run and Pass labels closer to the dumbbells and zoomed the axis to make the shift more visible.

## Synthesis and future refinements

The strongest element of the approach was pairing a focused team level story with a single league context chart. This prevented information overload while still letting the viewer see where Baltimore sits relative to everyone else. The first chart worked well because it uses familiar categorical structure and supports fast comparison across situations. The third chart worked well because it emphasizes change instead of forcing the viewer to compare bars across two separate groups.

If I had more time, the first refinement would be adding a small amount of context that explains sample size. For example, tooltips already include play counts, but I could add a light annotation showing total plays per facet so the viewer can distinguish a strong tendency from a small sample. I would also consider an interactive control that allows the user to switch the focal team from the Ravens to another team while keeping the design consistent. Finally, I would validate the findings with an expert viewer, such as a coach or an analyst, to confirm that the distance buckets and efficiency measure align with how football decision making is typically evaluated.
